# 04 · Drought in cattle country

**Milestone 1, item 1.6** · see `docs/MILESTONES.md`

## What we're looking for

The first link in the D17 chain: **drought → less grazing → cow sell-off → smaller herd
→ higher beef prices.** Notebook 03 showed the sell-off and the shrinking herd; this
notebook measures the drought, month by month, so we can check it came first.

Drought only matters for beef where the cattle are, so we build one number for
**"drought in cattle country"**, decided together (D26):

1. **States:** the states that together hold about **75% of U.S. beef cows**.
2. **Weights:** each state counts in proportion to its beef cows on **January 1, 2019**,
   the herd's peak before the drought. Fixed weights keep a state that *lost* cows to
   drought from counting less afterwards, which would hide the drought.
3. **Measures:**
   - **% of area in severe drought or worse (D2–D4):** the plain-English headline.
   - **DSCI** (Drought Severity and Coverage Index, 0–500): one number combining how
     much area is in drought and how severe it is. Used for the statistics in Milestone 4.
4. **Period:** January 2010 onward, matching the herd data (D24), so the 2011 drought is
   included. Weekly maps are averaged to months.

### The drought categories

The U.S. Drought Monitor classifies every part of the country each week:

| Category | Meaning |
|---|---|
| D0 | Abnormally dry (not yet drought) |
| D1 | Moderate drought |
| D2 | Severe drought |
| D3 | Extreme drought |
| D4 | Exceptional drought |

**Steps**
1. Setup
2. Beef cows by state (NASS) and the state weights
3. Download weekly drought statistics for each state (Drought Monitor)
4. Load and check the drought data
5. Monthly averages and the cattle-weighted measures
6. Chart: drought vs. cow slaughter
7. Save the tables

## Step 1 · Setup

The NASS key is needed again for the state beef cow counts. The Drought Monitor service
needs no key.

In [ ]:
import json
import os
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv()
load_dotenv(env_path)
NASS_API_KEY = os.getenv("NASS_API_KEY", "").strip()

PROJECT_ROOT = Path(env_path).parent if env_path else Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw" / f"drought_{date.today():%Y%m%d}"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

pd.set_option("display.max_rows", 100)
print("Project root:", PROJECT_ROOT)
print("NASS key found:", bool(NASS_API_KEY))

## Step 2 · Beef cows by state, and the state weights

**What we're looking for:** each state's beef cows on January 1, 2019, ranked from most to
least. We keep states, largest first, until they add up to at least 75% of the U.S. total.
Each kept state's weight is its share of the kept states' cows, so the weights add to 1.

NASS sometimes reports small states together as an "OTHER STATES" row. That isn't a real
state, so it's dropped before ranking.

In [ ]:
WEIGHT_YEAR = 2019
COVERAGE_TARGET = 75   # percent of U.S. beef cows

resp = requests.get("https://quickstats.nass.usda.gov/api/api_GET/", params={
    "key": NASS_API_KEY, "format": "JSON",
    "short_desc": "CATTLE, COWS, BEEF - INVENTORY",
    "agg_level_desc": "STATE", "source_desc": "SURVEY",
    "reference_period_desc": "FIRST OF JAN", "year": str(WEIGHT_YEAR),
}, timeout=120)
if resp.status_code != 200:
    raise RuntimeError(f"NASS failed ({resp.status_code}): {resp.text[:300]}")
(RAW_DIR / "nass_beef_cows_by_state.json").write_text(resp.text)

states = pd.DataFrame(resp.json()["data"])
states = states[states["domain_desc"] == "TOTAL"].copy()
states["beef_cows"] = pd.to_numeric(states["Value"].str.replace(",", "", regex=False), errors="coerce")
us_total = states["beef_cows"].sum()   # U.S. total, including any grouped "other states"

# NASS sometimes adds an "OTHER STATES" row (FIPS 98/99) for small states it groups together;
# it isn't a real state, so it can't be matched to drought data
states = states[~states["state_name"].str.upper().str.startswith("OTHER")
                & ~states["state_fips_code"].astype(str).isin(["98", "99"])]
states = (states[["state_name", "state_alpha", "state_fips_code", "beef_cows"]]
          .dropna().sort_values("beef_cows", ascending=False).reset_index(drop=True))

states["share_of_us_pct"] = states["beef_cows"] / us_total * 100
states["cumulative_pct"] = states["share_of_us_pct"].cumsum()

n_keep = int((states["cumulative_pct"] < COVERAGE_TARGET).sum()) + 1
cattle_states = states.head(n_keep).copy()
cattle_states["weight"] = cattle_states["beef_cows"] / cattle_states["beef_cows"].sum()

print(f"{n_keep} states hold {cattle_states['share_of_us_pct'].sum():.1f}% of U.S. beef cows ({WEIGHT_YEAR})")
cattle_states.assign(beef_cows_M=cattle_states["beef_cows"] / 1e6)[
    ["state_name", "state_alpha", "state_fips_code", "beef_cows_M", "share_of_us_pct", "cumulative_pct", "weight"]
].round(3)

### How to read it
One row per kept state: beef cows on January 1, 2019 (millions), its share of the U.S.
total, the running total, and its **weight** in the cattle-country measure (the weights
add up to 1).

### What it means
_Fill in after the first run: how many states, which ones, and how heavily Texas weighs._

## Step 3 · Download weekly drought statistics

For each state we ask the Drought Monitor for the weekly **percent of the state's area**
in each category, using its *traditional* (cumulative) format: the D1 column means "D1 or
worse", D2 means "D2 or worse", and so on. That makes **D2 directly the "% in severe
drought or worse"** we want.

We also download the **lower 48 states (CONUS)** as a national comparison.

One request per state; each saved untouched to the dated raw folder.

In [ ]:
USDM = "https://usdmdataservices.unl.edu/api"
START = "1/1/2010"
END = f"{date.today():%m/%d/%Y}"

def usdm_get(area, aoi):
    url = f"{USDM}/{area}/GetDroughtSeverityStatisticsByAreaPercent"
    params = {"aoi": aoi, "startdate": START, "enddate": END, "statisticsType": 1}
    resp = requests.get(url, params=params, headers={"Accept": "application/json"}, timeout=120)
    if resp.status_code != 200:
        raise RuntimeError(f"Drought Monitor failed for {aoi} ({resp.status_code}): {resp.text[:300]}")
    return resp.json()

drought_raw = {}
for _, st in cattle_states.iterrows():
    fips = str(st["state_fips_code"]).zfill(2)
    drought_raw[st["state_alpha"]] = usdm_get("StateStatistics", fips)
    print(f"{st['state_alpha']}: {len(drought_raw[st['state_alpha']]):>5} weeks")
drought_raw["CONUS"] = usdm_get("USStatistics", "conus")
print(f"CONUS: {len(drought_raw['CONUS']):>5} weeks")

(RAW_DIR / "usdm_state_stats.json").write_text(json.dumps(drought_raw))

### How to read it
About **870 weeks** per state is what we expect (January 2010 to now). A state with far
fewer would need a closer look.

## Step 4 · Load and check

The service's field names can differ slightly in capitalization, so we lowercase them
first. The week's date comes from `validStart` (the first day the map applies), falling
back to `mapDate`.

**Two checks:**
- In cumulative format each category includes the worse ones, so for every week
  **D0 ≥ D1 ≥ D2 ≥ D3 ≥ D4** must hold. If it doesn't, the data came back in the other
  (categorical) format and we'd need to add the columns up instead.
- All percentages should be between 0 and 100.

In [ ]:
def to_weekly(rows, area):
    df = pd.DataFrame(rows)
    df.columns = [col.lower() for col in df.columns]
    date_col = "validstart" if "validstart" in df.columns else "mapdate"
    raw_dates = df[date_col].astype(str)
    if raw_dates.str.fullmatch(r"\d{8}").all():
        df["week"] = pd.to_datetime(raw_dates, format="%Y%m%d")
    else:
        df["week"] = pd.to_datetime(raw_dates.str[:10])
    for col in ["d0", "d1", "d2", "d3", "d4"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["area"] = area
    return df[["area", "week", "d0", "d1", "d2", "d3", "d4"]].sort_values("week")

weekly = pd.concat([to_weekly(rows, area) for area, rows in drought_raw.items()], ignore_index=True)

cumulative_ok = ((weekly["d0"] >= weekly["d1"] - 0.01) & (weekly["d1"] >= weekly["d2"] - 0.01)
                 & (weekly["d2"] >= weekly["d3"] - 0.01) & (weekly["d3"] >= weekly["d4"] - 0.01)).mean() * 100
in_range = weekly[["d0", "d1", "d2", "d3", "d4"]].stack().between(0, 100).mean() * 100

print(f"Weeks where D0 >= D1 >= ... >= D4: {cumulative_ok:.1f}%  (should be 100)")
print(f"Values between 0 and 100:          {in_range:.1f}%  (should be 100)")
weekly.groupby("area")["week"].agg(["min", "max", "count"])

### How to read it
Both checks should print **100%**. The table lists each area's first and last week and
how many weeks it has.

## Step 5 · Monthly averages and the cattle-weighted measures

For each state and month:
- **d2_plus_pct:** average % of the state's area in severe drought or worse (the D2 column).
- **dsci:** the Drought Severity and Coverage Index, the sum of the five cumulative
  columns (D0 + D1 + D2 + D3 + D4). That's the same as weighting each category's area by
  1 for D0 up to 5 for D4, which is how the Drought Monitor defines it. It runs from 0
  (no dryness anywhere) to 500 (the whole state in exceptional drought).

Then the **cattle-country** value for each month is the weighted average across the kept
states, using the 2019 beef cow weights from Step 2.

In [ ]:
weekly["d2_plus_pct"] = weekly["d2"]
weekly["dsci"] = weekly[["d0", "d1", "d2", "d3", "d4"]].sum(axis=1)
weekly["month"] = weekly["week"].dt.to_period("M").dt.to_timestamp()

monthly_state = weekly.groupby(["area", "month"])[["d2_plus_pct", "dsci"]].mean().reset_index()

weights = cattle_states.set_index("state_alpha")["weight"]
in_states = monthly_state[monthly_state["area"].isin(weights.index)].copy()
in_states["w"] = in_states["area"].map(weights)

def weighted(group, col):
    return (group[col] * group["w"]).sum() / group["w"].sum()

drought = pd.DataFrame({
    "cattle_d2_plus_pct": in_states.groupby("month").apply(weighted, "d2_plus_pct", include_groups=False),
    "cattle_dsci": in_states.groupby("month").apply(weighted, "dsci", include_groups=False),
})
conus = monthly_state[monthly_state["area"] == "CONUS"].set_index("month")
drought["conus_d2_plus_pct"] = conus["d2_plus_pct"]
drought["conus_dsci"] = conus["dsci"]
drought = drought.rename_axis("date")

yearly_drought = drought.groupby(drought.index.year).mean().round(1)
yearly_drought

### How to read it
Yearly averages of the monthly values:
- **cattle_d2_plus_pct:** average % of cattle country in severe drought or worse.
- **cattle_dsci:** the same, as the 0–500 index.
- **conus_…:** the same measures for the lower 48 states as a whole, for comparison.

### What it means
_Fill in after the first run: which years were worst for cattle country, and how the
2011 and 2020–2023 droughts compare._

## Step 6 · Chart: drought vs. cow slaughter

**What we're looking for:** the order of events. If drought drives the sell-off, the
drought line should rise **before or alongside** beef cow slaughter (the 12-month average
from notebook 03), and fall back before slaughter does.

This reads `data/processed/nass_slaughter_monthly.csv`, so **notebook 03 must have been
run in full first**.

In [ ]:
slaughter = pd.read_csv(PROCESSED_DIR / "nass_slaughter_monthly.csv", parse_dates=["date"]).set_index("date")

fig, ax = plt.subplots(figsize=(11, 5))
ax.fill_between(drought.index, drought["cattle_d2_plus_pct"], color="#C08A1E", alpha=0.35,
                label="Cattle country in severe drought or worse (%)")
ax.plot(drought.index, drought["conus_d2_plus_pct"], color="#8A6414", linewidth=1, linestyle="--",
        label="Lower 48 states, same measure (%)")
ax.set_ylabel("% of area in D2-D4 drought")
ax.set_ylim(0, 100)

ax2 = ax.twinx()
ax2.plot(slaughter.index, slaughter["beef_cows_slaughtered_12mo"] / 12 / 1e3, color="#9A3B2E", linewidth=2.4,
         label="Beef cows slaughtered (12-month average, thousands/month)")
ax2.set_ylabel("Beef cows slaughtered, thousands per month", color="#9A3B2E")

lines = ax.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
labels = ax.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
ax.legend(lines, labels, loc="upper left", fontsize=8, frameon=True)
ax.set_title("Drought in cattle country and the beef cow sell-off", loc="left", fontweight="bold")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

### How to read it
- **Shaded area (left axis):** the % of cattle country in severe drought or worse, each
  month, weighted by beef cows.
- **Dashed line (left axis):** the same measure for the whole lower 48, for comparison.
- **Red line (right axis):** beef cows slaughtered per month, as a 12-month average.

### What it means
_Fill in after the first run._

## Step 7 · Save the tables

Two processed files (committed to git, D9, D12):
- `drought_cattle_country_monthly.csv`: the monthly measures.
- `drought_state_weights.csv`: which states were used and their weights, so the measure
  can be reproduced and explained on the app's methods page.

In [ ]:
drought_path = PROCESSED_DIR / "drought_cattle_country_monthly.csv"
drought.round(3).reset_index().to_csv(drought_path, index=False)

weights_path = PROCESSED_DIR / "drought_state_weights.csv"
cattle_states.assign(weight_year=WEIGHT_YEAR).to_csv(weights_path, index=False)

for p in (drought_path, weights_path):
    print("Saved:", p.relative_to(PROJECT_ROOT))

## What's next and why

**Item 1.7: grocery category CPIs** (`05_grocery_categories.ipynb`), the last data item
in Milestone 1. It reuses notebook 01's BLS download to put beef in the context of the
whole grocery cart (D19). After that, Milestone 2 cleans and combines everything into one
monthly table.